# P8 — Collaborative Multi-Agent DevSecOps Code Audit
Local rule-based demonstration of Security → Remediation → QA. The example vulnerability is intentionally included for training.

In [62]:
from pathlib import Path
import os

# Correct Kaggle dataset root
BASE_DIR = Path(
    "/kaggle/input/datasets/babblu123/"
    "real-world-ml-genai-kaggle-complete/"
    "real_world_ml_genai_lab"
)

DATA_DIR = BASE_DIR / "datasets"
ex_dir = BASE_DIR / "examples"

print("Dataset directory:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())
print("Examples directory:", ex_dir)
print("Examples directory exists:", ex_dir.exists())

# Check that both source files exist
vulnerable_path = ex_dir / "p8_vulnerable_app.py"
secure_path = ex_dir / "p8_secure_app.py"

print("Available example files:",
      [p.name for p in ex_dir.iterdir()] if ex_dir.exists() else [])

if not vulnerable_path.exists() or not secure_path.exists():
    raise FileNotFoundError(
        "One or both P8 example files are missing. "
        "Check the Available example files list above."
    )

# Load the examples
vulnerable = vulnerable_path.read_text(encoding="utf-8")
secure = secure_path.read_text(encoding="utf-8")

print("\n--- VULNERABLE APPLICATION ---")
print(vulnerable)

print("\n--- SECURE APPLICATION ---")
print(secure)

Dataset directory: /kaggle/input/datasets/babblu123/real-world-ml-genai-kaggle-complete/real_world_ml_genai_lab/datasets
Dataset exists: True
Examples directory: /kaggle/input/datasets/babblu123/real-world-ml-genai-kaggle-complete/real_world_ml_genai_lab/examples
Examples directory exists: True
Available example files: ['p8_secure_app.py', 'p8_vulnerable_app.py']

--- VULNERABLE APPLICATION ---
# Intentionally vulnerable training example. Do not deploy.
import sqlite3

def find_user(conn, username):
    query = "SELECT * FROM users WHERE username = '" + username + "'"
    return conn.execute(query).fetchall()


--- SECURE APPLICATION ---
# Safer parameterized query example.
def find_user(conn, username):
    return conn.execute("SELECT * FROM users WHERE username = ?", (username,)).fetchall()



In [63]:
def security_scanner(code_text):
    findings=[]
    if " + username" in code_text and "SELECT" in code_text.upper():
        findings.append({"severity":"high","rule":"SQL injection risk","evidence":"User input concatenated into SQL query","recommendation":"Use a parameterized query."})
    return findings
def remediation_agent(code_text,findings):
    patched=code_text.replace('query = "SELECT * FROM users WHERE username = \'" + username + "\'"',
                              'query = "SELECT * FROM users WHERE username = ?"')
    patched=patched.replace('return conn.execute(query).fetchall()','return conn.execute(query, (username,)).fetchall()')
    return patched
def qa_agent(code_text):
    try: compile(code_text,"candidate.py","exec"); syntax_ok=True
    except SyntaxError: syntax_ok=False
    return {"syntax_ok":syntax_ok,"parameterized_query":("execute(query, (username,))" in code_text)}
findings=security_scanner(vulnerable)
patch=remediation_agent(vulnerable,findings)
qa=qa_agent(patch)
print("Security findings:",findings)
print("Proposed patch:\n",patch)
print("QA:",qa)

Security findings: [{'severity': 'high', 'rule': 'SQL injection risk', 'evidence': 'User input concatenated into SQL query', 'recommendation': 'Use a parameterized query.'}]
Proposed patch:
 # Intentionally vulnerable training example. Do not deploy.
import sqlite3

def find_user(conn, username):
    query = "SELECT * FROM users WHERE username = ?"
    return conn.execute(query, (username,)).fetchall()

QA: {'syntax_ok': True, 'parameterized_query': True}


## Agent trace and limitations
Document scanner findings, proposed diff, QA checks, and human approval. This is a teaching prototype, not a complete security scanner; never execute untrusted code outside a sandbox.